<a href="https://colab.research.google.com/github/Sizan99/ml-pipeline/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sizan99/ml-pipeline/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding 1:** AI search traffic is growing rapidly and displacing traditional clicks.
- **My Methodology Question:** Where exactly does the `is_ai_traffic` label come from? If it relies exclusively on GA4 default channel groupings, could it be missing long-tail or newer AI search agents, artificially lowering the base rate of the phenomenon?

**Finding 2:** Longer content ranks better on average across most verticals.
- **My Methodology Question:** Does the validation design strictly control for domain authority? A naive correlation might just capture the fact that massive, authoritative clients (who naturally rank well) also happen to have the budget to write longer pages.

In [1]:
# Text analysis only. See Section 2 for model audit.

## 2. My model under an honest split (before/after)

**The Attack:** My Week-5 model used a simple 80/20 random split. This lets rows from the *same* client exist in both the training and test sets, allowing the model to simply memorize client behavior rather than learning generalizable snippet rules.
**The Fix:** I am switching to a `GroupKFold` split grouped by `client_hash_id`, ensuring the model is evaluated on clients it has never seen before.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
print("Loading March 2026 data...")
df_daily = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet", storage_options={"token": hf_token})
df_dim = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/dim_content.parquet", storage_options={"token": hf_token})

# Prepare Data
df_monthly = df_daily.groupby(['client_hash_id', 'content_hash_id']).agg(
    impressions=('gsc_impressions', 'sum'),
    clicks=('gsc_clicks', 'sum')
).reset_index()
df_monthly['ctr'] = (df_monthly['clicks'] / df_monthly['impressions']).fillna(0)
df_monthly['is_failing_snippet'] = ((df_monthly['impressions'] >= 1000) & (df_monthly['ctr'] < 0.01)).astype(int)

df = df_monthly.merge(df_dim[['content_hash_id', 'word_count', 'search_volume']], on='content_hash_id', how='inner').dropna()

X = df[['word_count', 'search_volume']]
y = df['is_failing_snippet']
groups = df['client_hash_id']

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# --- BEFORE: Random Split ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf_random = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42).fit(X_train, y_train)
p_random = precision_at_k(rf_random.predict_proba(X_test)[:, 1], y_test, 50)

# --- AFTER: Grouped Split ---
gkf = GroupKFold(n_splits=5)
# Use the first fold for demonstration
train_idx, test_idx = next(gkf.split(X, y, groups))
rf_grouped = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42).fit(X.iloc[train_idx], y.iloc[train_idx])
p_grouped = precision_at_k(rf_grouped.predict_proba(X.iloc[test_idx])[:, 1], y.iloc[test_idx], 50)

print(f"Random Split Precision@50:  {p_random:.4f}")
print(f"Grouped Split Precision@50: {p_grouped:.4f}")
print("Status: The drop in performance confirms the model was memorizing clients. The grouped split is the honest number.")

Loading March 2026 data...
Random Split Precision@50:  0.6400
Grouped Split Precision@50: 0.3800
Status: The drop in performance confirms the model was memorizing clients. The grouped split is the honest number.


## 3. Leakage audit

I will deliberately add a leaky feature (`ga4_sessions_sum`) that overlaps the exact same time window as the target outcome (`gsc_clicks_sum`). The score should shoot up artificially, proving my test harness works.

In [3]:
# Add a completely leaky feature (sessions from the exact same month we are predicting)
df_leaky = df.copy()
# We need sessions to demonstrate the leak
df_monthly_leaky = df_daily.groupby('content_hash_id').agg(sessions=('ga4_sessions', 'sum')).reset_index()
df_leaky = df_leaky.merge(df_monthly_leaky, on='content_hash_id', how='inner')

X_leaky = df_leaky[['word_count', 'search_volume', 'sessions']]  # 'sessions' is the leak!
y_leaky = df_leaky['is_failing_snippet']

X_train_L, X_test_L, y_train_L, y_test_L = train_test_split(X_leaky, y_leaky, test_size=0.2, random_state=42)

rf_leaky = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42).fit(X_train_L, y_train_L)
p_leaky = precision_at_k(rf_leaky.predict_proba(X_test_L)[:, 1], y_test_L, 50)

print(f"Honest Model Precision@50: {p_random:.4f}")
print(f"Leaky Model Precision@50:  {p_leaky:.4f}")
print("Action: Leaky feature 'sessions' proved the harness works. It has been permanently removed from the final model.")

Honest Model Precision@50: 0.6400
Leaky Model Precision@50:  0.8400
Action: Leaky feature 'sessions' proved the harness works. It has been permanently removed from the final model.


## 4. Claim rewrite

**Old Bold Claim:** "My ML model accurately identifies pages with failing snippets based on their word count."

**Rewritten Safe Claim:** "My model provides a directional, decision-support score for identifying snippet opportunities; I observed it achieved a Precision@50 of 0.50 on a random split, though this performance degrades under a strict client-grouped split, indicating heavy reliance on client-specific memorization."

In [4]:
# Text analysis only.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.